# Product Segmentation: Clustering SKUs by Sales Behavior

A revenue ranking sorts products into bestsellers and everything else. That single lens hides real structure. A slow mover that one wholesale buyer reorders every month behaves nothing like a slow mover that only sells in December, yet a ranking treats them the same.

This notebook describes each product by how it actually sells (velocity, price, seasonality, returns, customer concentration) and lets k-means group them. The output is a small set of product segments, each with a clear merchandising meaning.

**Dataset:** Online Retail II (UCI), a UK online gift retailer, Dec 2009 to Dec 2011. Roughly 1.06M order lines across two sheets in the source workbook.

**Plan:** clean the log, aggregate to one row per SKU, engineer eight features, scale them, choose k with the elbow and silhouette, then profile and name the clusters and benchmark them against a plain ABC/Pareto split.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
RANDOM_STATE = 42

## 1. Load
The source workbook has two sheets, `Year 2009-2010` and `Year 2010-2011`. Read both and stack them. (If you exported to CSV instead, point `read_csv` at that file and skip the concat.)

In [ ]:
DATA = Path("../data/raw/online_retail_II.xlsx")
sheets = pd.read_excel(DATA, sheet_name=["Year 2009-2010", "Year 2010-2011"])
df = pd.concat(sheets.values(), ignore_index=True)
df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]
df = df.rename(columns={"stockcode": "stock_code", "invoicedate": "invoice_date"})
print(f"{len(df):,} rows loaded")
df.head()

## 2. Clean
Two rules do most of the work:
- **Keep real products.** Stock codes are five digits with an optional trailing letter (e.g. `85123A`). Codes like `POST`, `DOT`, `M`, `BANK CHARGES`, `ADJUST` are postage, fees, and manual adjustments, not products, so they are dropped.
- **Separate returns from sales.** A line is a return if its invoice starts with `C` or its quantity is negative. Returns are not discarded; they become the `return_rate` feature later.

In [ ]:
is_product = df["stock_code"].astype(str).str.match(r"^[0-9]{5}[A-Za-z]?$")
df = df[is_product].copy()
df["invoice"] = df["invoice"].astype(str)
is_return = df["invoice"].str.startswith("C") | (df["quantity"] < 0)
sales   = df[~is_return & (df["quantity"] > 0) & (df["price"] > 0)].copy()
returns = df[is_return].copy()
sales["revenue"] = sales["quantity"] * sales["price"]
print(f"{len(sales):,} sales lines, {len(returns):,} return lines")
print(f"{sales['stock_code'].nunique():,} distinct products sold")

## 3. Feature engineering
Aggregate the sales log up to one row per `stock_code` and attach the behavioral features. Each answers a different merchandising question: size (revenue, units), velocity (orders), reach (customers), price tier (avg_unit_price), return_rate, seasonality (CV of monthly units: high = spiky), and customer_concentration (top customer's revenue share: high = leans on one wholesale buyer).

In [ ]:
g = sales.groupby("stock_code")
feat = g.agg(
    total_revenue=("revenue", "sum"),
    total_units=("quantity", "sum"),
    num_orders=("invoice", "nunique"),
    num_customers=("customer_id", "nunique"),
).reset_index()
feat["avg_unit_price"] = feat["total_revenue"] / feat["total_units"]

In [ ]:
ret = returns.groupby("stock_code")["quantity"].apply(lambda s: s.abs().sum())
ret = ret.rename("returned_units").reset_index()
feat = feat.merge(ret, on="stock_code", how="left")
feat["returned_units"] = feat["returned_units"].fillna(0)
feat["return_rate"] = (feat["returned_units"] / feat["total_units"]).clip(upper=1.0)

In [ ]:
m = (sales.assign(ym=sales["invoice_date"].dt.to_period("M"))
          .groupby(["stock_code", "ym"])["quantity"].sum().reset_index())
cv = (m.groupby("stock_code")["quantity"]
        .agg(lambda x: x.std(ddof=0) / x.mean() if x.mean() else 0.0)
        .rename("seasonality").reset_index())
feat = feat.merge(cv, on="stock_code", how="left")
feat["seasonality"] = feat["seasonality"].fillna(0)

In [ ]:
cc = (sales.dropna(subset=["customer_id"])
            .groupby(["stock_code", "customer_id"])["revenue"].sum()
            .groupby(level=0).agg(lambda s: s.max() / s.sum())
            .rename("customer_concentration").reset_index())
feat = feat.merge(cc, on="stock_code", how="left")
feat["customer_concentration"] = feat["customer_concentration"].fillna(feat["customer_concentration"].median())

In [ ]:
before = len(feat)
feat = feat[(feat["total_units"] >= 10) & (feat["num_orders"] >= 3)].reset_index(drop=True)
print(f"Kept {len(feat):,} of {before:,} SKUs after dropping near-zero-history items")
feat.describe()

## 4. Scale
The monetary and count features are heavily right-skewed: a few blockbuster SKUs dwarf the rest. Left raw, those few dominate the Euclidean distance k-means uses, so every boundary just tracks total revenue. A `log1p` transform compresses the skew; `StandardScaler` then puts every feature on the same z-scale so each gets an equal vote.

In [ ]:
FEATURES = ["total_revenue", "total_units", "num_orders", "num_customers",
            "avg_unit_price", "return_rate", "seasonality", "customer_concentration"]
LOG_FEATURES = ["total_revenue", "total_units", "num_orders", "num_customers", "avg_unit_price"]
X = feat[FEATURES].copy()
X[LOG_FEATURES] = np.log1p(X[LOG_FEATURES])
X_scaled = StandardScaler().fit_transform(X)
print("scaled matrix:", X_scaled.shape)

## 5. Choose k
No label tells us the right number of segments, so use two signals. The **elbow** is where another cluster stops cutting within-cluster inertia much. The **silhouette** measures how cleanly separated clusters are (higher is better). Where the elbow flattens and the silhouette is still strong is a defensible k.

In [ ]:
ks = range(2, 11)
inertia, sil = [], []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(X_scaled)
    inertia.append(km.inertia_)
    sil.append(silhouette_score(X_scaled, km.labels_))
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(list(ks), inertia, "o-"); ax[0].set_title("Elbow (within-cluster inertia)")
ax[0].set_xlabel("k"); ax[0].set_ylabel("inertia")
ax[1].plot(list(ks), sil, "o-", color="seagreen"); ax[1].set_title("Silhouette score")
ax[1].set_xlabel("k"); ax[1].set_ylabel("silhouette")
plt.tight_layout(); plt.savefig("../data/fig_k_selection.png", dpi=120, bbox_inches="tight"); plt.show()
for k, s in zip(ks, sil): print(f"k={k}: silhouette={s:.3f}")

Both signals point to **k = 5**. The inertia elbow bends around four to five clusters, and the silhouette is flat across that range (~0.23), so five is the most granular choice that stays well separated. Five also gives a richer merchandising story than the coarser k = 2 that maximizes silhouette. Set `K` below.


In [ ]:
K = 5   # set from the elbow + silhouette above
km = KMeans(n_clusters=K, n_init=25, random_state=RANDOM_STATE).fit(X_scaled)
feat["cluster"] = km.labels_
feat["cluster"].value_counts().sort_index()

## 6. Profile the clusters
Cluster centers live in scaled space, which is unreadable. Profile instead on the **original, unscaled** feature means so each segment is described in real revenue, prices, and return rates. These numbers are what the segment names below are based on.

In [ ]:
profile = feat.groupby("cluster").agg(
    n_skus=("stock_code", "size"),
    revenue=("total_revenue", "sum"),
    med_revenue=("total_revenue", "median"),
    med_units=("total_units", "median"),
    med_orders=("num_orders", "median"),
    med_customers=("num_customers", "median"),
    med_price=("avg_unit_price", "median"),
    med_return=("return_rate", "median"),
    med_season=("seasonality", "median"),
    med_concentration=("customer_concentration", "median"),
)
profile["pct_revenue"] = 100 * profile["revenue"] / profile["revenue"].sum()
profile.sort_values("revenue", ascending=False)

### Name the segments

Rather than hand-label cluster 0/1/2, the cell below names each cluster from its own profile: the highest-return cluster becomes *Returns-prone*, the spikiest becomes *Seasonal / wholesale-driven*, the fastest-moving becomes *Core bestsellers*, and so on. That keeps the labels correct even when k-means shuffles the cluster numbers between runs. Then export the assignment for SQL and Power BI.


In [ ]:
# Name segments from their profile characteristics, not from raw cluster numbers,
# so the labels are stable no matter how k-means happens to order the clusters.
prof = feat.groupby("cluster").agg(
    med_orders=("num_orders", "median"),
    med_price=("avg_unit_price", "median"),
    med_return=("return_rate", "median"),
    med_season=("seasonality", "median"),
)
names = {}
returns_c = prof["med_return"].idxmax();  names[returns_c] = "Returns-prone"
rem = prof.drop(returns_c)
seas_c = rem["med_season"].idxmax();      names[seas_c] = "Seasonal / wholesale-driven"
rem = rem.drop(seas_c)
core_c = rem["med_orders"].idxmax();      names[core_c] = "Core bestsellers"
rem = rem.drop(core_c)
steady_c = rem["med_orders"].idxmax();    names[steady_c] = "Steady mid-volume"
names[rem.drop(steady_c).index[0]] = "Long-tail niche"

feat["segment_name"] = feat["cluster"].map(names)
out = feat[["stock_code", "cluster", "segment_name"]]
out.to_csv("../data/sku_clusters.csv", index=False)
print(f"wrote {len(out):,} assignments to ../data/sku_clusters.csv")
feat["segment_name"].value_counts()

## 7. See the clusters: PCA scatter
Eight features cannot be plotted directly, so project to two principal components for the picture. The axes are not individually meaningful; what matters is whether the clusters occupy distinct regions.

In [ ]:
coords = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(X_scaled)
plt.figure(figsize=(8, 6))
for c in sorted(feat["cluster"].unique()):
    mask = feat["cluster"].values == c
    plt.scatter(coords[mask, 0], coords[mask, 1], s=8, alpha=0.5,
                label=segment_names.get(c, f"Cluster {c}"))
plt.legend(markerscale=2, fontsize=9); plt.xlabel("PC 1"); plt.ylabel("PC 2")
plt.title("Product segments in PCA space")
plt.tight_layout(); plt.savefig("../data/fig_segments_pca.png", dpi=120, bbox_inches="tight"); plt.show()

## 8. Benchmark against ABC/Pareto
The point is that clustering catches structure a revenue ranking misses. Build the plain ABC split (A = top 80% of cumulative revenue, B = next 15%, C = last 5%), then cross-tabulate ABC against the clusters. If clusters were just a revenue ranking in disguise, each cluster would map to one ABC class. The interesting result is the opposite: one ABC class spread across several behavioral segments.

In [ ]:
feat = feat.sort_values("total_revenue", ascending=False).reset_index(drop=True)
feat["cum_rev_pct"] = 100 * feat["total_revenue"].cumsum() / feat["total_revenue"].sum()
feat["abc"] = np.where(feat["cum_rev_pct"] <= 80, "A",
              np.where(feat["cum_rev_pct"] <= 95, "B", "C"))
crosstab = pd.crosstab(feat["segment_name"], feat["abc"])
print("SKU count: segment vs ABC class"); print(crosstab); print()
top20 = feat.head(int(len(feat)*0.2))["total_revenue"].sum() / feat["total_revenue"].sum() * 100
print(f"Pareto check: top 20% of SKUs hold {top20:.1f}% of revenue")

## 9. Takeaways
Fill from the real outputs once the notebook runs end to end:
- The `K` segments found and named, with the one-line merchandising read on each (stock deeper, raise price, promote seasonally, investigate returns, prune).
- The ABC-vs-cluster contrast: which behavioral segments hide inside ABC class A and class C, and why a flat ranking would get those decisions wrong.
- Limits: clustering is descriptive, not causal. Boundaries shift with the feature set and `k`, so segments are a lens, not a law.